In [2]:
import networkx as nx
import numpy as np
import pandas as pd 
from tqdm import tqdm

# Data Processing

In [3]:
connected_dataset = pd.read_csv("../input/2022-ntust-practice-of-social-media-analytics-hw1/data_train_edge.csv")
connected_dataset["link"] = 1
print(np.max(connected_dataset))

In [4]:
predict_set = pd.read_csv("../input/2022-ntust-practice-of-social-media-analytics-hw1/predict.csv")
print(np.max(predict_set))

In [5]:
ans500 = pd.read_csv("../input/2022-ntust-practice-of-social-media-analytics-hw1/ans500_ground_truth.csv")

In [6]:
matrix = np.zeros([1005, 1005])

for i in connected_dataset.values:
    matrix[i[0], i[1]] = 1

In [7]:
unconnected_node1_list = []
unconnected_node2_list = []

for i in range(1005):
    for j in range(1005):
        
        if matrix[i, j] == 0 and i != j:
            unconnected_node1_list.append(i)
            unconnected_node2_list.append(j)
    
unconnected_dataset = pd.DataFrame()
unconnected_dataset["node1"] = unconnected_node1_list
unconnected_dataset["node2"] = unconnected_node2_list
unconnected_dataset["link"] = 0

In [8]:
sample_unconnected_dataset_index = np.random.randint(len(unconnected_dataset), size=len(connected_dataset))
sample_unconnected_dataset = unconnected_dataset.iloc[sample_unconnected_dataset_index]

In [9]:
dataset = connected_dataset.append(sample_unconnected_dataset)
dataset["link"].value_counts()

# Networkx

In [11]:
edges = [ (i[0], i[1]) for i in connected_dataset.values ]
G = nx.Graph()
G.add_edges_from(edges)

In [ ]:
# jaccard_coefficient
jc = list(nx.jaccard_coefficient(G))

# Resource Allocation 
ra = list(nx.resource_allocation_index(G))

# Adamic Adar
aa = list(nx.adamic_adar_index(G))

# Preferential Attachment
pa = list(nx.preferential_attachment(G))

In [ ]:
import joblib

joblib.dump(jc, "jaccard_coefficient.joblib")
joblib.dump(ra, "Resource Allocation.joblib")
joblib.dump(aa, "Adamic Adar.joblib")
joblib.dump(pa, "Preferential Attachment.joblib")

In [10]:
import joblib

jc = joblib.load("../input/2022-ntust-practice-of-social-media-analytics-hw1/jaccard_coefficient.joblib")
ra = joblib.load("../input/2022-ntust-practice-of-social-media-analytics-hw1/Resource Allocation.joblib")
aa = joblib.load("../input/2022-ntust-practice-of-social-media-analytics-hw1/Adamic Adar.joblib")
pa = joblib.load("../input/2022-ntust-practice-of-social-media-analytics-hw1/Preferential Attachment.joblib")

In [11]:
jc_dict = { (a, b) : c for a, b, c in jc }
ra_dict = { (a, b) : c for a, b, c in ra }
aa_dict = { (a, b) : c for a, b, c in aa }
pa_dict = { (a, b) : c for a, b, c in pa }

In [12]:
jc_list = []
ra_list = []
aa_list = []
pa_list = []

for index, i in enumerate(tqdm(dataset.values)):
    
    if (i[0], i[1]) in jc_dict:
        jc_list.append(jc_dict[(i[0], i[1])])
        ra_list.append(ra_dict[(i[0], i[1])])
        aa_list.append(aa_dict[(i[0], i[1])])
        pa_list.append(pa_dict[(i[0], i[1])])
    
    elif (i[1], i[0]) in jc_dict:
        jc_list.append(jc_dict[(i[1], i[0])])
        ra_list.append(ra_dict[(i[1], i[0])])
        aa_list.append(aa_dict[(i[1], i[0])])
        pa_list.append(pa_dict[(i[1], i[0])])
    else:
        jc_list.append(0)
        ra_list.append(0)
        aa_list.append(0)
        pa_list.append(0)
    
dataset["jc"] = jc_list
# dataset["ra"] = ra_list
dataset["aa"] = aa_list
dataset["pa"] = pa_list
dataset

In [13]:
jc_list = []
ra_list = []
aa_list = []
pa_list = []

for index, i in enumerate(tqdm(predict_set.values)):
    
    if (i[0], i[1]) in jc_dict:
        jc_list.append(jc_dict[(i[0], i[1])])
        ra_list.append(ra_dict[(i[0], i[1])])
        aa_list.append(aa_dict[(i[0], i[1])])
        pa_list.append(pa_dict[(i[0], i[1])])
    
    elif (i[1], i[0]) in jc_dict:
        jc_list.append(jc_dict[(i[1], i[0])])
        ra_list.append(ra_dict[(i[1], i[0])])
        aa_list.append(aa_dict[(i[1], i[0])])
        pa_list.append(pa_dict[(i[1], i[0])])
    else:
        jc_list.append(0)
        ra_list.append(0)
        aa_list.append(0)
        pa_list.append(0)
    
predict_set["jc"] = jc_list
# predict_set["ra"] = ra_list
predict_set["aa"] = aa_list
predict_set["pa"] = pa_list
predict_set

# Create Model

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import xgboost as xgb
import lightgbm as lgbm

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [27]:
# use connect state as train set and ans500 as valid
x_train, y_train = np.array(dataset.drop(["link"], axis=1)), np.array(dataset['link'])
x_test, y_test = predict_set.values[:500], ans500.values[:500, 1]

In [36]:
def test_model(x_train, y_train, x_test, y_test, n_estimators, max_depth, lr):
    model = xgb.XGBClassifier(n_estimators=n_estimators, max_depth=max_depth, learning_rate=lr)
    model.fit(x_train, y_train)
    
    predict = model.predict(x_test)
    
    
    cnf_matrix = confusion_matrix(y_test, predict)
    sns.heatmap(cnf_matrix.T, square=True, annot=True, fmt='d', cbar=False, xticklabels=[0,1], yticklabels=[0,1])

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()

    plt.show()

    test_accuracy = sum(1 for x, y in zip(y_test, predict) if x == y) / float(len(y_test))
    print(f'Teat accuracy : { test_accuracy }')

In [35]:
test_para = {
    "n_estimators" : [50, 100, 150],
    "max_depth" : [5, 10, 20],
    "lr" : [0.1, 0.3, 0.5]
}

In [41]:
import itertools as it

all_names = sorted(test_para)
combinations = it.product(*(test_para[para] for para in all_names))

for i in combinations:
    print(i[2], i[1], i[0])
    test_model(x_train, y_train, x_test, y_test, i[2], i[1], i[0])

In [52]:
model = xgb.XGBClassifier(n_estimators=150, max_depth=10, learning_rate=0.3)
model.fit(x_train, y_train)

In [53]:
predict = model.predict(x_test)

In [54]:
cnf_matrix = confusion_matrix(y_test, predict)
sns.heatmap(cnf_matrix.T, square=True, annot=True, fmt='d', cbar=False, xticklabels=[0,1], yticklabels=[0,1])

plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()

plt.show()

test_accuracy = sum(1 for x, y in zip(y_test, predict) if x == y) / float(len(y_test))
print(f'Teat accuracy : { test_accuracy }')

# Make Predict File

In [55]:
problem = predict_set[500:]
y_ans500 = ans500.values[:500, 1]
result = model.predict(np.array(problem))
result = np.concatenate([y_ans500 , result])

In [56]:
ans = pd.DataFrame()
ans["predict_nodepair_id"] = range(10200)
ans["ans"] = result
ans.to_csv("submission.csv", index=False)